In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 1. Generate synthetic X
num_subjects = 200
num_features = 5

X_raw = torch.randn(num_subjects, num_features)


# 2. Add intercept column
ones_column = torch.ones(num_subjects, 1)

X = torch.cat((ones_column, X_raw), dim=1)
# X shape: (500, 6)


# 3. Define true beta
# 第一個是 intercept
beta_true = torch.tensor([
    1.0,   # intercept
    2.0,   # X1 effect
    -1.5,  # X2 effect
    0.5,   # X3 effect
    0.0,   # X4 no effect
    3.0    # X5 effect
]).reshape(-1,1)


# 4. Generate Y according to linear model
noise = torch.randn(num_subjects,1) * 0.5

Y_raw = X @ beta_true + noise

In [2]:
# 確認模擬資料是對的
beta = torch.linalg.solve(
    X.T @ X,
    X.T @ Y_raw
)

print(beta)

tensor([[ 1.0338],
        [ 2.0110],
        [-1.5079],
        [ 0.5220],
        [ 0.0102],
        [ 2.9512]])


In [3]:
from sklearn.model_selection import train_test_split

# 建立 index
indices = torch.arange(num_subjects)

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42
)

# 切資料
X_train_raw = X_raw[train_idx]
X_test_raw = X_raw[test_idx]

Y_train = Y_raw[train_idx]
Y_test = Y_raw[test_idx]

In [4]:
# Train跟test都要加截距項
ones_train = torch.ones(X_train_raw.shape[0],1)
ones_test = torch.ones(X_test_raw.shape[0],1)


X_train = torch.cat(
    (ones_train, X_train_raw),
    dim=1
)

X_test = torch.cat(
    (ones_test, X_test_raw),
    dim=1
)

In [5]:
# 先把Y也加入X中，要讓attention matrix有Y的資訊
X_Y_train = torch.cat(
    (X_train, Y_train),
    dim=1
)

In [6]:
# 原始程式碼貼上(不訓練attention)

# Transpose data so the 6 features plus Y act as the "sequence" 
X_Y_features = X_Y_train.t()

# 5. Define the projection dimension (d_k)
d_k = 32
W_Q = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

W_K = nn.Linear(
    X_Y_features.shape[1],
    d_k,
    bias=False
)

# 6. Project features into Query (Q) and Key (K) spaces
Q = W_Q(X_Y_features)
K = W_K(X_Y_features)

# 7. Compute the raw attention scores
scores = torch.matmul(
    Q,
    K.transpose(-2,-1)
)

# 8. Scale by sqrt(d_k) and apply Softmax row-wise
attention_matrix = F.softmax(
    scores / (d_k ** 0.5),
    dim=-1
)

In [7]:
print("Attention Matrix Shape:", attention_matrix.shape)
print("\nAttention Matrix:\n", attention_matrix)

Attention Matrix Shape: torch.Size([7, 7])

Attention Matrix:
 tensor([[0.0726, 0.1348, 0.0706, 0.0993, 0.1084, 0.1489, 0.3653],
        [0.1497, 0.1239, 0.1217, 0.1128, 0.2096, 0.1539, 0.1283],
        [0.1513, 0.1184, 0.1720, 0.1820, 0.0866, 0.1814, 0.1082],
        [0.1508, 0.1373, 0.1275, 0.2107, 0.2055, 0.1031, 0.0651],
        [0.3313, 0.0844, 0.1801, 0.0725, 0.1344, 0.1358, 0.0615],
        [0.1410, 0.1273, 0.1491, 0.1814, 0.1439, 0.1637, 0.0935],
        [0.0769, 0.0725, 0.0334, 0.1005, 0.4756, 0.1253, 0.1158]],
       grad_fn=<SoftmaxBackward0>)


In [8]:
# 把Y再從矩陣中拿掉
A = attention_matrix[:-1,:-1]

print("Attention Matrix Shape(拿掉Y):", A.shape)
print("\nAttention Matrix(拿掉Y):\n", A)

Attention Matrix Shape(拿掉Y): torch.Size([6, 6])

Attention Matrix(拿掉Y):
 tensor([[0.0726, 0.1348, 0.0706, 0.0993, 0.1084, 0.1489],
        [0.1497, 0.1239, 0.1217, 0.1128, 0.2096, 0.1539],
        [0.1513, 0.1184, 0.1720, 0.1820, 0.0866, 0.1814],
        [0.1508, 0.1373, 0.1275, 0.2107, 0.2055, 0.1031],
        [0.3313, 0.0844, 0.1801, 0.0725, 0.1344, 0.1358],
        [0.1410, 0.1273, 0.1491, 0.1814, 0.1439, 0.1637]],
       grad_fn=<SliceBackward0>)


In [9]:
# 矩陣乘上Y的變異數
var_y = torch.var(Y_train)

A_var_y = A * var_y

In [10]:
A_var_y

tensor([[1.0631, 1.9732, 1.0329, 1.4532, 1.5868, 2.1799],
        [2.1914, 1.8128, 1.7815, 1.6509, 3.0685, 2.2528],
        [2.2145, 1.7333, 2.5180, 2.6637, 1.2682, 2.6556],
        [2.2078, 2.0097, 1.8658, 3.0835, 3.0080, 1.5092],
        [4.8485, 1.2351, 2.6363, 1.0617, 1.9666, 1.9878],
        [2.0642, 1.8637, 2.1825, 2.6557, 2.1068, 2.3953]],
       grad_fn=<MulBackward0>)

In [11]:
X_train.T @ X_train

tensor([[ 1.6000e+02,  5.8334e+00,  3.6936e+00, -1.1815e+01, -4.6796e+00,
         -9.3578e-01],
        [ 5.8334e+00,  1.8537e+02,  9.1169e+00, -5.4665e+00,  1.6161e+01,
         -1.3263e+00],
        [ 3.6936e+00,  9.1169e+00,  1.5607e+02, -3.2913e+00,  1.0303e+00,
          3.8618e+00],
        [-1.1815e+01, -5.4665e+00, -3.2913e+00,  1.5592e+02, -7.4284e-03,
          1.9659e+00],
        [-4.6796e+00,  1.6161e+01,  1.0303e+00, -7.4284e-03,  1.5807e+02,
          1.0998e+00],
        [-9.3578e-01, -1.3263e+00,  3.8618e+00,  1.9659e+00,  1.0998e+00,
          1.4449e+02]])

In [12]:
# 放入OLS的公式解中，估計beta
#beta_attention = torch.linalg.solve((X_train.T @ X_train + A_var_y)/2,X_train.T @ Y_train)
p = X.shape[1]

I = torch.eye(
    p,
    dtype=X.dtype,
    device=X.device
)

beta_attention = torch.linalg.solve((X_train.T @ X_train - 0.1*A),X_train.T @ Y_train)

print(
    "Attention beta:",
    beta_attention
)

Attention beta: tensor([[ 1.0265e+00],
        [ 2.0180e+00],
        [-1.5116e+00],
        [ 5.3628e-01],
        [ 9.5364e-04],
        [ 2.9224e+00]], grad_fn=<LinalgSolveExBackward0>)


In [13]:
# 把算出來的係數套入驗證集
Y_pred_attention = X_test @ beta_attention

# 算MSE
mse_attention = torch.mean(
    (Y_test - Y_pred_attention)**2
)

In [14]:
# OLS的標準解法
beta_ols = torch.linalg.solve(
    X_train.T @ X_train,
    X_train.T @ Y_train
)

print(
    "OLS beta:",
    beta_ols
)

Y_pred_ols = X_test @ beta_ols

mse_ols = torch.mean(
    (Y_test - Y_pred_ols)**2
)



OLS beta: tensor([[ 1.0260e+00],
        [ 2.0176e+00],
        [-1.5120e+00],
        [ 5.3581e-01],
        [ 5.5409e-04],
        [ 2.9219e+00]])


In [15]:
print(
    "Attention MSE:",
    mse_attention.item()
)

Attention MSE: 0.237862229347229


In [16]:
print(
    "OLS MSE:",
    mse_ols.item()
)

OLS MSE: 0.23800404369831085
